# 06 — Results Analysis

Pull every metric written by the previous notebooks and build the final
comparison table that goes into the README. Honest interpretation:
for ICU mortality prediction with 6-qubit feature maps and N=200
training points, **the classical SVM is hard to beat**. The interesting
result is not that QSVM wins — it is *how close* the quantum kernels
get with so few qubits, and at what runtime cost.

In [ ]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from qml_healthcare.evaluation import figure_path, load_results
from qml_healthcare.pipeline import run_reports

warnings.filterwarnings("ignore")

In [ ]:
flat = run_reports()  # writes final_comparison.png, runtime_comparison.png, etc.
df = pd.DataFrame(flat).T
cols = ["accuracy", "balanced_accuracy", "roc_auc", "pr_auc", "f1", "train_seconds"]
df = df[[c for c in cols if c in df.columns]].sort_values("roc_auc", ascending=False)
df.style.format("{:.3f}")

## Cross-cutting observations

1. **Quantum advantage on this task is not real (yet).** The classical
   SVM, given the same K-feature, N-sample subset, performs at least as
   well as every quantum model — and on the full 91k-row dataset it pulls
   further ahead. This matches the broader QML literature: most quantum
   kernels demonstrated to date are simulable classically up to small
   qubit counts.

2. **Feature-map choice matters more than ansatz depth.** ZZ vs Pauli
   vs the custom RY/RZ ring map produce noticeably different ROC-AUC
   even at the same N. The block structure visible in the kernel
   heatmaps from notebook 03 maps directly onto downstream
   classification quality.

3. **Runtime is the killer.** QSVM kernel computation is O(N²) circuit
   evaluations; on a CPU simulator at N=200 each feature map takes
   minutes to hours. Real quantum hardware (with queue + shots) makes
   this orders of magnitude worse.

4. **Where this approach could matter.** The literature points to
   regimes where the quantum kernel is provably hard to simulate
   (Liu, Arunachalam & Temme, 2021). For practical ICU prediction,
   classical kernels are the right tool today — but the engineering
   stack (feature selection, PSD projection, fidelity estimation)
   carries over directly when those regimes become accessible.